# 10. 멀티모달 임베딩 — OCR 없이 스캔 페이지 찾기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/10_multimodal_embeddings.ipynb)

> 🧭 **스토리** — 총무팀 캐비닛에서 나온 규정집은 **스캔 이미지**뿐이고, 직원들은 영수증 사진과 차트 캡처도 함께 올린다. 06~08번처럼 글자를 뽑아내지 않고, **이미지를 그대로 벡터로 바꿔** 한국어 질문으로 찾을 수 있을까?
>
> 📖 **이론 연결** — 이론2 p71(멀티모달 임베딩: 이미지·텍스트 한 공간, 문서 특화 모델, EG2) · ⏱ **15분** · 🖥 **GPU 권장**
>
> GPU(T4)가 있으면 빠릅니다. 없어도 CPU로 돌아갑니다(느림).
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
09번과 같은 `sentence-transformers` 6.1 이상을 씁니다. 이번에는 EmbeddingGemma 2(EG2)의 **비전 인코더까지** 올립니다.
GPU가 없으면 이미지 수를 줄여(스캔 3쪽 + 이미지 2장) 진행할 수 있습니다. 환경변수 `NB_SMOKE=1`이면 자동으로 줄입니다.

In [ ]:
%pip -q install -U sentence-transformers transformers pillow

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-ai-api/data)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-ai-api/main/data/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ 검색 대상 이미지 8장 — 스캔 페이지 6장 + 페이지가 아닌 이미지 2장(영수증 사진, 막대그래프)
import os
from PIL import Image
import matplotlib.pyplot as plt

SMOKE = os.environ.get("NB_SMOKE") == "1"           # CPU에서 빨리 확인할 때: 스캔 3쪽만
pages = ["p2", "p4", "p5"] if SMOKE else ["p1", "p2", "p3", "p4", "p5", "p6"]
img_files = {p: f"scan/{p}.png" for p in pages}
img_files.update({"receipt": "images/receipt.png", "chart": "images/chart_allowance.png"})
images = {k: Image.open(fetch(v)).convert("RGB") for k, v in img_files.items()}

fig, axes = plt.subplots(1, len(images), figsize=(2.1 * len(images), 3))
for ax, (k, im) in zip(axes, images.items()):
    ax.imshow(im); ax.set_title(f"{k} {im.size[0]}×{im.size[1]}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

스캔 페이지는 약간 기울고 잡음이 섞여 있습니다(06번에서 본 그 파일). p2에는 국내 출장비 표, p3에는 해외 출장비 표, p4에는 연차·경조사 휴가 표, p5에는 경조금·교육비 표가 있습니다.

## 1. EG2 비전 포함 로드 🔵
09번에서는 `config_kwargs`로 비전·음성 부품을 모두 뺐습니다. 이번에는 **음성만 빼고** 비전은 켭니다(카드의 "Text and image" 설정, 약 440M).
float16은 금지입니다(값이 NaN이 되거나 조용히 망가짐). bf16을 지원하는 GPU(A100·L4 등)면 bf16, T4·CPU면 float32를 씁니다.

In [ ]:
import time, torch, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

EG_DTYPE = torch.bfloat16 if (HAS_GPU and torch.cuda.is_bf16_supported()) else torch.float32   # T4 → float32
t0 = time.time()
eg = SentenceTransformer("google/embeddinggemma-2", device=DEVICE,
                         model_kwargs={"dtype": EG_DTYPE},
                         config_kwargs={"audio_config": None})     # 텍스트+이미지. 전부 쓰려면 config_kwargs를 빼면 됩니다(740M)
n_params = sum(p.numel() for p in eg.parameters())
print(f"로드 {time.time()-t0:.1f}초 · dtype {EG_DTYPE} · 파라미터 {n_params/1e6:.0f}M · 차원 {eg.get_embedding_dimension()}")

💡 **해석** — 강사 환경에서 파라미터 439M이 찍혔습니다. 텍스트 전용(09번, 271M)보다 비전 인코더 약 170M이 늘었습니다. float32 기준 메모리는 약 1.8GB입니다.

## 2. 이미지를 벡터로 🔵
이미지는 `{"image": PIL이미지}` 딕셔너리로 넣습니다. **이미지에는 접두어(prompt_name)를 붙이지 않습니다.**
메모리를 아끼려고 한 장씩 넣습니다. 이미지 한 장은 기본 **280개의 시각 토큰(soft token)** 으로 요약됩니다.

In [ ]:
img_names = list(images)
img_vecs, rows = [], []
for k in img_names:
    t0 = time.time()
    v = eg.encode({"image": images[k]}, normalize_embeddings=True)      # (768,)
    img_vecs.append(v); rows.append({"이미지": k, "초": round(time.time() - t0, 2)})
IMG = np.stack(img_vecs)
print("이미지 행렬:", IMG.shape)
display(pd.DataFrame(rows).set_index("이미지").T)

💡 **해석** — 강사 노트북(CPU, 다른 작업이 없을 때)에서 첫 장은 약 2초(준비 비용), 나머지는 한 장에 약 0.3초였습니다. CPU가 바쁘면 몇 배씩 늘어납니다. 코랩 T4(bfloat16)에서는 첫 장 약 2.3초, 나머지는 한 장에 약 0.7초였습니다(2026-10-17). T4는 bfloat16 연산을 하드웨어로 지원하지 않아 GPU인데도 생각보다 빠르지 않습니다. 결과는 텍스트와 **같은 768차원 공간**의 벡터입니다. 그래서 텍스트 질문 벡터와 바로 코사인 유사도를 잴 수 있습니다.

## 3. 한국어 질문으로 이미지 찾기 🔵
질문은 텍스트이므로 09번과 똑같이 `prompt_name="SearchQuery"`를 붙입니다. 질문마다 정답 이미지를 정해 두고 **1등이 맞는지** 봅니다.

In [ ]:
queries = {   # 질문: 정답 이미지
    "과장 국내 출장 일비 표": "p2",
    "해외 출장 일비 달러": "p3",
    "경조사 휴가 일수": "p4",
    "연차휴가": "p4",
    "경조금 화환": "p5",
    "교육비 지원 한도": "p5",
    "호텔 영수증": "receipt",
    "직급별 출장비 막대그래프": "chart",
}
queries = {q: g for q, g in queries.items() if g in images}      # SMOKE 모드에서는 있는 이미지만

def rank_images(qs, M=None, prompt_name="SearchQuery"):
    M = IMG if M is None else M
    Q = eg.encode(list(qs), prompt_name=prompt_name, normalize_embeddings=True)
    S = eg.similarity(Q, M).numpy()
    out = []
    for (q, gold), row in zip(qs.items(), S):
        o = np.argsort(-row)
        out.append({"질문": q, "정답": gold,
                    "1등": f"{img_names[o[0]]} {row[o[0]]:.3f}", "2등": f"{img_names[o[1]]} {row[o[1]]:.3f}",
                    "3등": f"{img_names[o[2]]} {row[o[2]]:.3f}", "맞음": img_names[o[0]] == gold})
    return pd.DataFrame(out).set_index("질문"), S

tbl, S_img = rank_images(queries)
display(tbl)
print(f"hit@1 = {tbl['맞음'].mean():.2f}  ({tbl['맞음'].sum()}/{len(tbl)})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(S_img, cmap="Blues"); ax.set_xticks(range(len(img_names)), img_names); ax.set_yticks(range(len(queries)), list(queries))
for i in range(S_img.shape[0]):
    for j in range(S_img.shape[1]):
        ax.text(j, i, f"{S_img[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("질문 × 이미지 코사인 유사도"); plt.tight_layout(); plt.show()

💡 **해석** — 강사 환경(이미지 8장, 기본 280토큰) 측정값:

| 질문 | 정답 | 1등 (점수) | 결과 |
|---|---|---|---|
| 호텔 영수증 | receipt | receipt 0.782 | 맞음 |
| 직급별 출장비 막대그래프 | chart | chart 0.763 | 맞음 |
| 경조사 휴가 일수 · 연차휴가 | p4 | p4 0.697 · 0.717 | 맞음 |
| 경조금 화환 | p5 | p5 0.654 | 맞음 |
| 과장 국내 출장 일비 표 | p2 | chart 0.759 (p2 0.709는 2등) | 틀림 |
| 해외 출장 일비 달러 | p3 | chart 0.714 (p3 0.667은 2등) | 틀림 |
| 교육비 지원 한도 | p5 | receipt 0.662 | 틀림 |

- **hit@1 = 5/8**. OCR을 한 글자도 하지 않았는데 스캔 페이지에서 '연차휴가'·'경조금' 페이지를 찾았습니다. 영수증·차트처럼 **겉모습이 뚜렷한 이미지**는 점수 차이도 큽니다.
- 틀린 세 문항은 모두 **글자가 빽빽한 표 페이지**입니다. '일비'라는 말에 같은 내용을 그린 막대그래프가 끌려 왔습니다. 280토큰으로 A4 한 쪽을 요약하면 작은 글자 정보가 뭉개집니다. 8절의 🟡에서 토큰을 늘려 봅니다.
- 점수 범위(0.6~0.8)가 좁습니다. 1등과 2등 차이가 0.01~0.05인 경우가 많아 **임계값으로 자르기 어렵습니다**.

## 4. 텍스트와 이미지를 한 공간에서 🔵
09번의 텍스트 청크 13개도 같은 모델로 임베딩해 **이미지와 한 인덱스에** 넣어 봅니다. 질문 하나에 대해 이미지와 텍스트가 섞인 순위가 나옵니다.

In [ ]:
chunks = load_json("hanbit_chunks_gt.json")
TXT = eg.encode([c["text"] for c in chunks], prompt_name="Document", normalize_embeddings=True)
mixed_names = [f"[이미지] {k}" for k in img_names] + [f"[텍스트] {c['id']}" for c in chunks]
MIX = np.vstack([IMG, TXT])

for q in ["과장 국내 출장 일비 표", "호텔 영수증"]:
    s = eg.similarity(eg.encode([q], prompt_name="SearchQuery", normalize_embeddings=True), MIX).numpy()[0]
    o = np.argsort(-s)[:6]
    print(f"\n[{q}]")
    for r, j in enumerate(o, 1):
        print(f"  {r}. {mixed_names[j]:<22} {s[j]:.3f}")

S_all = eg.similarity(eg.encode(list(queries), prompt_name="SearchQuery", normalize_embeddings=True), MIX).numpy()
print(f"\n평균 유사도 — 질문↔이미지 {S_all[:, :len(img_names)].mean():.3f} · 질문↔텍스트 청크 {S_all[:, len(img_names):].mean():.3f}")

💡 **해석** — 강사 환경에서 "과장 국내 출장 일비 표"는 텍스트 청크 `s2_t1_domestic`(0.83)이 1등이었습니다. 정답 이미지 p2(0.71)보다 관련 없는 텍스트 청크들이 더 위에 섰습니다.
"호텔 영수증"은 영수증 이미지(0.78)가 1등이었습니다. 규정 텍스트에는 영수증 그 자체가 없기 때문입니다.
- 질문↔텍스트 점수가 질문↔이미지 점수보다 **전반적으로 높습니다**. 같은 공간이라도 모달리티마다 점수대가 다릅니다. 이것을 **모달리티 격차(modality gap)** 라고 합니다.
- 그래서 실무에서는 이미지 인덱스와 텍스트 인덱스를 **따로 검색해 각각 상위 몇 개씩** 가져오거나, 09번의 RRF처럼 **순위로** 합칩니다. 점수를 그대로 섞으면 한쪽만 올라옵니다.

### 09번 텍스트 검색과 비교하면
- 09번에서 EG2는 텍스트 청크로 규정 질문 hit@1 0.85, hit@3 1.00이었습니다. 이 노트북의 이미지 검색은 기본 설정에서 hit@1 5/8입니다. **같은 모델이라도 텍스트로 뽑아 둔 쪽이 더 정확합니다.**
- 대신 이미지 검색은 OCR·파싱 단계가 없습니다. 파싱이 실패하는 문서(손글씨, 도장, 복잡한 도면, 사진)나 **이미지 자체가 답**인 경우(영수증, 차트)에 강합니다.
- 문서 페이지 검색에 특화된 모델(ColPali 계열, Qwen3-VL-Embedding)은 이 격차를 줄입니다. 오늘은 설치가 간단하고 게이트가 없는 EG2만 씁니다(이론2 p71).

## 5. 바꿔서 실행 🟡
1. **영어로 물으면?** 같은 내용을 영어로 물어 한국어 질문과 비교하세요.

In [ ]:
queries_en = {
    "domestic business trip daily allowance table": "p2",
    "family event leave days": "p4",
    "hotel receipt": "receipt",
    "bar chart of allowance by job grade": "chart",
}
queries_en = {q: g for q, g in queries_en.items() if g in images}
tbl_en, _ = rank_images(queries_en)
display(tbl_en)

💡 **해석** — 강사 환경(기본 280토큰)에서 영어 질문 4개 중 3개가 맞았습니다(p4 0.635, receipt 0.710, chart 0.691). "domestic business trip daily allowance table"은 한국어 질문과 똑같이 chart(0.680)가 1등, 정답 p2(0.648)는 3등이었습니다.
영어 질문의 점수는 한국어 질문보다 0.03~0.07 낮았지만 **순위는 거의 같습니다**. 한국어 페이지를 영어로 찾을 수 있고, 한국어 질문이라서 불리하지도 않았습니다.

2. **시각 토큰을 늘리면?** `max_soft_tokens`는 이미지 한 장을 몇 개의 토큰으로 요약할지 정합니다(70~1120, 기본 280). 글자가 빽빽한 페이지에 1120을 줘 봅니다. 시간은 늘어납니다.

In [ ]:
def embed_images(max_soft_tokens):
    t0 = time.time()
    M = np.stack([eg.encode({"image": images[k]}, normalize_embeddings=True,
                            processing_kwargs={"image": {"max_soft_tokens": max_soft_tokens}}) for k in img_names])
    return M, time.time() - t0

rows = []
for mst in [280, 1120]:        # ← 70 이나 560 도 넣어 보세요
    M, sec = embed_images(mst)
    t, _ = rank_images(queries, M=M)
    rows.append({"max_soft_tokens": mst, "이미지 전체 임베딩(초)": round(sec, 1), "hit@1": f"{t['맞음'].sum()}/{len(t)}",
                 "틀린 질문": ", ".join(t.index[~t["맞음"]]) or "-"})
display(pd.DataFrame(rows).set_index("max_soft_tokens"))

💡 **해석** — 강사 환경(이미지 8장, CPU) 측정값:

| max_soft_tokens | 8장 임베딩 시간(CPU 한가할 때) | (CPU 바쁠 때) | hit@1 |
|---|---|---|---|
| 280 (기본) | 3.3초 | 15.5초 | 5/8 |
| 1120 | 20.2초 | 177초 | **8/8** |

- 토큰을 4배로 늘리자 틀렸던 표 페이지 세 개(p2·p3·p5)를 모두 찾았습니다. 작은 글자가 많은 문서 페이지에는 **해상도(토큰 수)가 곧 품질**입니다.
- 대가는 시간입니다. CPU에서 6~11배 느려졌습니다. 코랩 T4(bfloat16) 실측은 280토큰 5.6초 · 1120토큰 56.5초였고, hit@1은 5/8 → 7/8(과장 국내 출장 일비 표만 여전히 틀림)이었습니다(2026-10-17). 페이지가 수만 장이면 이 차이가 서버비가 됩니다.
- 다만 p2와 chart의 점수 차는 0.744 대 0.743으로 아슬아슬했습니다. 8개 질문으로 내린 결론이라는 점을 기억하세요.

### 🤔 생각해보기

**멀티모달 검색은 OCR+텍스트 검색을 대체할까, 보완할까?** 오늘 결과(정확도·속도·모달리티 격차)를 근거로 말해 보세요.

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- 오늘 결과만 보면 **보완**입니다. 같은 EG2라도 텍스트 청크 검색(09번 hit@1 0.85)이 이미지 페이지 검색(기본 5/8)보다 정확했습니다. 표 안의 숫자를 답해야 하는 RAG라면 결국 글자가 필요합니다. 페이지 이미지를 찾아도 LLM이 답하려면 OCR이나 VLM(07번)이 한 번 더 필요합니다.
- 반대로 **이미지 자체가 답인 경우**(영수증·차트·사진·도면)나 파싱이 깨지는 문서에서는 멀티모달 검색이 유일한 방법일 수 있습니다.
- 현실적인 설계는 **두 인덱스를 함께 두는 것**입니다. 텍스트 인덱스(파싱 결과)와 이미지 인덱스(페이지 이미지)를 따로 검색해 순위로 합칩니다. 페이지가 많고 파싱 품질이 나쁘면 문서 특화 멀티모달 모델(ColPali 계열)을 검토합니다.
- 비용도 따집니다. 시각 토큰을 늘리면 정확도는 오르지만 임베딩 시간이 몇 배로 늘어납니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 멀티모달 임베딩은 이미지와 텍스트를 한 벡터 공간에 넣는다. OCR 없이 한국어 질문으로 스캔 페이지를 찾을 수 있다.
- 이미지에는 접두어를 붙이지 않는다. 질문 텍스트에만 `prompt_name`을 붙인다.
- 글자가 빽빽한 페이지는 시각 토큰(`max_soft_tokens`)을 늘려야 잘 찾는다. 대신 느려진다.
- 모달리티마다 점수대가 다르다. 텍스트와 이미지를 섞을 때는 점수가 아니라 순위로 합친다.

> 다음 노트북: **11_mini_rag_local_vs_api — 근거를 대고 답하는 도우미 v1**